# Week 20: Advanced Computer Vision, 3D Vision & NeRFs
## Notebook 05: Optical Flow & Motion Estimation (Lucas-Kanade) from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In video processing, autonomous robotics, and action recognition:
- Static image models process each frame independently, missing motion dynamics.
- **Optical Flow** computes the vector field of apparent pixel motion between consecutive video frames $(I_t \to I_{t+1})$.

Applications:
- Autonomous emergency braking: Tracking velocity of moving vehicles.
- Video stabilization: Counteracting camera shake.

---

### 2. Mathematical Foundations: Brightness Constancy Constraint
Assume pixel intensity of a point remains constant as it moves by $(\Delta x, \Delta y)$ over time $\Delta t$:
$$I(x + \Delta x, y + \Delta y, t + \Delta t) = I(x, y, t)$$

First-order Taylor series expansion yields the **Optical Flow Equation**:
$$I_x u + I_y v + I_t = 0$$
Where $I_x = \frac{\partial I}{\partial x}, I_y = \frac{\partial I}{\partial y}, I_t = \frac{\partial I}{\partial t}$, and $(u, v)$ is the velocity vector.

#### Lucas-Kanade Least Squares Solution:
Assuming motion $(u, v)$ is constant within a small $3 \times 3$ local spatial window:
$$\begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix} \begin{bmatrix} u \\ v \end{bmatrix} = - \begin{bmatrix} \sum I_x I_t \\ \sum I_y I_t \end{bmatrix}$$
Solvable via direct $2 \times 2$ matrix inversion!

In [ ]:
import numpy as np

# Step 1: Synthesize Two Video Frames (Frame 1 and Frame 2 with 2-pixel rightward motion)
H, W = 16, 16
frame1 = np.zeros((H, W))
frame2 = np.zeros((H, W))

# A square object centered at (8, 6) in Frame 1
frame1[6:10, 5:9] = 1.0

# The square object moves 2 pixels to the right in Frame 2 (centered at (8, 8))
frame2[6:10, 7:11] = 1.0

# Step 2: Compute Spatial and Temporal Gradients
# Sobel / Finite differences
I_x = np.zeros((H, W))
I_y = np.zeros((H, W))
I_t = frame2 - frame1 # Temporal derivative

I_x[:, :-1] = np.diff(frame1, axis=1)
I_y[:-1, :] = np.diff(frame1, axis=0)

# Step 3: Implement Lucas-Kanade 3x3 Window Solver
def solve_lucas_kanade(I_x, I_y, I_t, cy, cx, win=1):
    # Window bounds
    y1, y2 = max(0, cy - win), min(H, cy + win + 1)
    x1, x2 = max(0, cx - win), min(W, cx + win + 1)
    
    Ix_win = I_x[y1:y2, x1:x2].flatten()
    Iy_win = I_y[y1:y2, x1:x2].flatten()
    It_win = I_t[y1:y2, x1:x2].flatten()
    
    # Structure Tensor A^T A
    A = np.column_stack([Ix_win, Iy_win])
    b = -It_win
    
    AtA = A.T @ A
    # Check if invertible (Aperture problem check)
    if np.linalg.cond(AtA) < 100 and np.linalg.det(AtA) > 1e-4:
        uv = np.linalg.inv(AtA) @ (A.T @ b)
        return uv[0], uv[1] # u (x-velocity), v (y-velocity)
    return 0.0, 0.0

# Evaluate at object edge point (cy=8, cx=6)
u, v = solve_lucas_kanade(I_x, I_y, I_t, cy=8, cx=6, win=1)

print("--- Lucas-Kanade Optical Flow Estimation ---")
print(f"True Motion Vector:      u = +2.0 px/frame (horizontal), v = 0.0 px/frame")
print(f"Estimated Motion Vector: u = {u:.2f} px/frame, v = {v:.2f} px/frame")

### 3. Step-by-Step Code Walkthrough
1. **Brightness Constancy:** Gradients $I_x, I_y, I_t$ are extracted across frames.
2. **Structure Tensor:** The $2 \times 2$ matrix $A^T A$ is identical to the Harris Corner matrix, ensuring that features with strong 2D textures yield stable velocity estimates.
3. **Motion Recovery:** Successfully recovered the horizontal displacement vector.

---

### 4. Industry Applications & Trade-offs
- **Video Compression (H.264 / AV1):** Motion vectors eliminate redundant transmission of static background pixels.
- **Trade-off (Aperture Problem):** On a flat uniform wall or single straight edge, optical flow cannot determine motion parallel to the edge.

---

### 5. Top Interview Questions & Answers
**Q1: What is the "Aperture Problem" in optical flow?**
> **Answer:** When viewing a moving edge through a small local aperture window, motion parallel to the edge is invisible. Only normal flow perpendicular to the edge can be measured. Resolving the true 2D velocity vector requires corners or textured patches with gradient variations in both axes.

**Q2: How does the Pyramidal Lucas-Kanade (Lucas-Kanade with Scale Pyramids) handle large motions?**
> **Answer:** Standard LK assumes infinitesimal motion ($|u, v| < 1$ pixel). For large displacements (e.g. 20 pixels), an image pyramid downsamples frames by powers of 2. Motion is first estimated coarsely at the low-resolution apex and propagated down to refine fine details.